# Day-1 Probe — Constant Response Baseline

**This submission is a diagnostic, not an attempt to score well.** It emits one fixed
generic Bengali doctor response for all 1,000 test rows. It answers two questions that
nothing else can answer as cheaply:

### 1. Which submission column name is correct?
Kaggle's **Data** tab says `id,output`; the **Overview** tab says `id,doctor_response`.
Both are Kaggle pages, so rule precedence cannot settle it, and there is no
`sample_submission.csv`. We submit with `output` (the Data tab outranks Overview for
file schema, and it matches the `train.csv` column). If the submission errors, flip
`SUBMISSION_COLUMN` below and resubmit.

### 2. Is BERTScore rescaled with a baseline?
Measured locally, a constant string scores **~0.454** composite when BERTScore is
unrescaled, because unrescaled BERTScore sits in a ~0.68–0.70 band for any fluent
in-domain Bengali text.

| Leaderboard result | Meaning | Action |
|---|---|---|
| **≈ 0.42 – 0.48** | BERTScore is **unrescaled** — it barely discriminates | PLAN.md holds: optimize Token F1 / ROUGE-L |
| **≈ 0.10 – 0.20** | BERTScore **is rescaled** — it becomes highly discriminative | **Rewrite PLAN.md §0**; reweight toward semantic quality |

Record the result in `PREDICTIONS.md` and `PROGRESS.md` either way.

In [ ]:
import glob
import os
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────
# Kaggle Data tab specifies exactly two columns: id,output
SUBMISSION_COLUMN = "output"

# The competition input directory is not always named after the competition slug,
# so glob for it rather than hardcoding a path.
if os.path.isdir("/kaggle/input"):
    print("/kaggle/input contains:", os.listdir("/kaggle/input"))

candidates = sorted(glob.glob("/kaggle/input/**/test.csv", recursive=True)) + [
    "../DATA/COMPETITION_PROVIDED_DATA/test.csv",
    "DATA/COMPETITION_PROVIDED_DATA/test.csv",
]
test_path = next((p for p in candidates if os.path.exists(p)), None)
if test_path is None:
    raise FileNotFoundError(
        "test.csv not found. If running on Kaggle, attach the competition data:\n"
        "  Notebook editor -> Add Input -> Competitions -> Nascenia AI Hackathon\n"
        f"Searched: {candidates}"
    )

test = pd.read_csv(test_path)
print(f"loaded {test_path}  shape={test.shape}  columns={list(test.columns)}")

## The constant response

Built from the highest document-frequency tokens in `train.csv` outputs, kept fluent and
clinically safe. It deliberately contains **no specific diagnosis** — measured on held-out
data, retrieval of a specific-but-wrong real response scores **0.431** while generic
boilerplate scores **0.454**. Confident specifics that miss cost more precision than they
gain in recall.

Design notes:
- Opens with **হেলো** — 76.23% of reference responses do.
- Self-identifies as **নাসেনিয়া ডক** — appears in ~48% of references.
- Targets ~110 tokens against a reference median of 93. Measured length sensitivity shows
  score rising to ~120 tokens then saturating, so slight overshoot is safer than undershoot.
- Includes the high-frequency closing scaffolding (আশা করি … ধন্যবাদ).

In [ ]:
CONSTANT_RESPONSE = (
    "হেলো, নাসেনিয়া ডকে আপনাকে স্বাগতম। আপনার অনুসন্ধানের জন্য ধন্যবাদ। "
    "আমি আপনার প্রশ্নটি দেখেছি এবং আপনার উদ্বেগ বুঝতে পেরেছি। "
    "আমি যথাসাধ্য আপনাকে সাহায্য করার চেষ্টা করব। "
    "আপনার বর্ণনা করা লক্ষণগুলো বিভিন্ন কারণে হতে পারে এবং এর সঠিক কারণ নির্ণয়ের জন্য "
    "একটি বিস্তারিত পরীক্ষা প্রয়োজন। তাই আমি আপনাকে একজন বিশেষজ্ঞ ডাক্তারের সাথে "
    "সরাসরি পরামর্শ করার পরামর্শ দিচ্ছি। প্রয়োজনীয় পরীক্ষা-নিরীক্ষা করানো উচিত এবং "
    "রিপোর্ট অনুযায়ী চিকিৎসা শুরু করা যেতে পারে। এই সময়ে পর্যাপ্ত বিশ্রাম নিন, "
    "প্রচুর পানি পান করুন এবং স্বাস্থ্যকর খাবার খান। ডাক্তারের পরামর্শ ছাড়া কোনো ওষুধ "
    "সেবন করবেন না। যদি সমস্যা বাড়তে থাকে বা তীব্র হয়, তাহলে দেরি না করে দ্রুত "
    "ডাক্তারের শরণাপন্ন হন। আশা করি এই উত্তরটি আপনাকে সাহায্য করবে। "
    "আরও কোনো প্রশ্ন থাকলে নির্দ্বিধায় জিজ্ঞাসা করতে পারেন। ধন্যবাদ।"
)

print(f"length: {len(CONSTANT_RESPONSE)} chars, {len(CONSTANT_RESPONSE.split())} whitespace tokens")
print(f"reference median: 93 tokens / ~594 chars")
print()
print(CONSTANT_RESPONSE)

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    SUBMISSION_COLUMN: CONSTANT_RESPONSE,
})

submission.to_csv("submission.csv", index=False)
print(f"wrote submission.csv  shape={submission.shape}  columns={list(submission.columns)}")

## Sanity checks

A malformed submission wastes a slot and teaches nothing. Verify before trusting the score.

In [ ]:
check = pd.read_csv("submission.csv")

problems = []
if len(check) != 1000:
    problems.append(f"expected 1000 rows, got {len(check)}")
if list(check.columns) != ["id", SUBMISSION_COLUMN]:
    problems.append(f"unexpected columns: {list(check.columns)}")
if check["id"].duplicated().any():
    problems.append("duplicate ids present")
if set(check["id"]) != set(test["id"]):
    problems.append("id set does not match test.csv")
if check[SUBMISSION_COLUMN].isna().any():
    problems.append("null responses present")
if (check[SUBMISSION_COLUMN].astype(str).str.strip() == "").any():
    problems.append("empty responses present")

# round-trip check: Bengali must survive the CSV write intact
if check[SUBMISSION_COLUMN].iloc[0] != CONSTANT_RESPONSE:
    problems.append("text changed on CSV round-trip (encoding issue)")

print(f"rows          {len(check)}")
print(f"columns       {list(check.columns)}")
print(f"unique ids    {check['id'].nunique()}")
print(f"encoding OK   {check[SUBMISSION_COLUMN].iloc[0] == CONSTANT_RESPONSE}")
print()
print("❌ " + "; ".join(problems) if problems else "✅ all checks passed")

check.head(3)

---

## After submitting — record the result

1. **`PREDICTIONS.md`** — fill row 1: predicted dev composite `~0.454`, actual public LB, delta.
2. **`PROGRESS.md`** — new entry: which column name worked, and what the score implies
   about BERTScore rescaling.
3. **`PLAN.md`** — **only if** the score lands near 0.10–0.20. That means BERTScore is
   rescaled, §0's central premise inverts, and the strategy must be reweighted toward
   semantic quality rather than lexical overlap.